In [1]:
import sys
sys.path.append("/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/src")

from technical_analysis.backtest import run_backtest, evaluate_run_performance
from technical_analysis.strategy import get_default_params, get_all_strategies

# Get Test DF

In [2]:
import pandas as pd
import os

test_dir = "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/"
ln_volume = True
test_filepaths = [os.path.join(
    test_dir, x) for x in os.listdir(test_dir) if str(ln_volume) in x]

# test_filepaths = [
#     "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-15_ln=False.csv",
#     "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-14_ln=False.csv",
#     "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-13_ln=False.csv",
#     "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-12_ln=False.csv", 
#     "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-11_ln=False.csv",
# ]



df_list = [pd.read_csv(fp) for fp in test_filepaths]
test_df = pd.concat(df_list, ignore_index=True)

# Apply Strategy

In [3]:
params_dict = get_default_params()
strategies = get_all_strategies()

In [4]:
"""
Possible Exit Reason:
- Take_Profit
- Stop_Loss
- Time_Limit_48h
- Data_Cutoff_End_of_file
- Stop_Loss_Simultaneous_Hit (TP and SL hit at the same time)
"""

eval_dict_ls = []
for name, fn in strategies.items():
    buy_df = fn(test_df, params_dict)
    backtest_df = run_backtest(
        buy_df, 
        params_dict,
    )
    eval_dict = evaluate_run_performance(backtest_df, params_dict)
    eval_dict = {"strategy": name, **eval_dict}
    eval_dict_ls.append(eval_dict)

In [5]:
sliced_columns = ["strategy", 'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours']
pd.DataFrame(eval_dict_ls)[sliced_columns]

,strategy,win_rate_adjusted,lose_rate_adjusted,timeout_rate_adjusted,profit_factor,avg_holding_hours
0,baseline,46.50,29.82,23.68,1.52,35.8
1,shift_v1,42.98,32.84,24.17,1.16,35.1
2,shift_v2,47.92,34.03,18.06,0.99,36.4
3,kotegawa,50.56,49.44,0.00,0.79,34.9
4,kotegawa_v2,50.82,49.18,0.00,0.99,36.3
5,kotegawa_v3,41.67,58.33,0.00,0.67,43.0
6,kotegawa_v4,66.67,33.33,0.00,1.34,37.7
7,momentum_filter,25.74,53.47,20.79,0.40,39.8


In [12]:
fn = strategies["kotegawa"]
buy_df = fn(test_df, params_dict)
backtest_df = run_backtest(
    buy_df, 
    params_dict,
)

In [15]:
from collections import Counter
Counter(backtest_df["exit_reason"])

Counter({'Take_Profit': 45,
         'Stop_Loss': 44,
         'Stop_Loss_Simultaneous_Hit': 7,
         'Data_Cutoff_End_of_File': 6})

In [18]:
backtest_df[backtest_df["exit_reason"] == "Stop_Loss"]

,ticker,buy_time,buy_price,exit_price,close_price,pnl,return,exit_reason,hours_held,buy_idx,sell_date
1,AEHR,2026-05-18 09:00:00,86.6000,84.002000,81.0001,-2.598000,-0.03,Stop_Loss,48,6085,2026-05-20 09:00:00
3,AMBA,2026-05-29 09:00:00,74.5100,72.274700,72.5351,-2.235300,-0.03,Stop_Loss,10,11758,2026-05-29 19:00:00
9,BFAM,2026-05-06 09:00:00,68.6500,66.590500,70.0550,-2.059500,-0.03,Stop_Loss,48,29881,2026-05-08 09:00:00
11,BSX,2026-05-27 09:00:00,51.0350,49.503950,49.0200,-1.531050,-0.03,Stop_Loss,48,36332,2026-05-29 09:00:00
12,CDW,2026-05-06 09:00:00,110.1350,106.830950,106.8100,-3.304050,-0.03,Stop_Loss,48,42808,2026-05-08 09:00:00
19,CTRI,2026-05-07 11:00:00,34.1500,33.125500,34.5000,-1.024500,-0.03,Stop_Loss,30,59894,2026-05-08 17:00:00
21,CVNA,2026-05-12 04:00:00,74.7900,72.546300,70.3900,-2.243700,-0.03,Stop_Loss,48,61378,2026-05-14 04:00:00
24,DXPE,2026-05-07 09:00:00,156.2000,151.514000,156.1700,-4.686000,-0.03,Stop_Loss,30,72814,2026-05-08 15:00:00
29,GKOS,2026-05-26 11:00:00,113.9600,110.541200,110.1750,-3.418800,-0.03,Stop_Loss,48,98416,2026-05-28 11:00:00
34,IBP,2026-05-07 09:00:00,240.5400,233.323800,219.9900,-7.216200,-0.03,Stop_Loss,31,112983,2026-05-08 16:00:00


In [5]:
sliced_columns = ["strategy", 'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours']
pd.DataFrame(eval_dict_ls)[sliced_columns]

,strategy,win_rate_adjusted,lose_rate_adjusted,timeout_rate_adjusted,profit_factor,avg_holding_hours
0,baseline,45.78,29.22,25.00,1.55,35.7
1,shift_v1,42.21,33.69,24.10,1.12,35.0
2,shift_v2,47.62,35.71,16.67,0.80,37.2
3,kotegawa,50.56,49.44,0.00,0.79,34.9
4,momentum_filter,25.51,59.18,15.31,0.41,40.2


In [8]:
sliced_columns = ["strategy", 'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours']
pd.DataFrame(eval_dict_ls)[sliced_columns]

,strategy,win_rate_adjusted,lose_rate_adjusted,timeout_rate_adjusted,profit_factor,avg_holding_hours
0,baseline,37.65,30.62,31.73,1.48,35.6
1,shift_v1,35.39,33.90,30.71,1.11,35.0
2,shift_v2,51.02,20.41,28.57,2.30,41.7
3,kotegawa,76.92,23.08,0.00,2.85,34.4
4,momentum_filter,25.00,70.83,4.17,0.31,39.6


In [7]:
pd.DataFrame(eval_dict_ls).columns

Index(['strategy', 'z_score_threshold', 'body_ratio_threshold',
       'profit_target', 'stop_loss_target', 'max_held_hours', 'total_trades',
       'win_rate', 'lose_rate', 'timeout_rate', 'cutoff_ratet',
       'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours'],
      dtype='object')

In [11]:
df = buy_df.copy()
(df["ema_20"] - df["close"]) / df["ema_20"]

0        0.009442
1        0.008351
2        0.009556
3        0.009253
4        0.010899
           ...   
65671   -0.001120
65672   -0.004546
65673   -0.002699
65674    0.003741
65675    0.003385
Length: 65676, dtype: float64